# Feedforward Meshing

The purpose of this notebook is to test different methods of feedforward point-cloud reconstruction. The primary methods tested so far are as follows:
- **VGGT-X:** transformer network with global aggregation module trained on intrinsic / extrinsic prediction
- **MapAnything:** builds on VGGT to impose metric scale reconstruction. Can take some degree of ground-truth data to improve the predictions / resultant pointcloud

Future improvements for integration:
- Loop closure (VGGT-Long / VGGT-SLAM)
- Sub-map alignment

Current testing to combine methods of frame sampling with mapanything, pushed into meshing. If this works will then integrate to improve the splatter module.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
from matplotlib import pyplot as plt
import numpy as np

import pyvista as pv

# Splatter module for information about video
from collab_splats.wrapper import Splatter
from collab_splats.utils.visualization import CAMERA_KWARGS, MESH_KWARGS, VIZ_KWARGS, visualize_splat

# Import optical flow module
from optical_flow import OpticalFlowFrameSelector, create_selection_summary_plot

# Import MapAnything utils
from mapanything_utils import run_mapanything_pipeline

## Step 1: Load Configuration from Splatter

Load dataset configuration using Splatter's config system.

In [ ]:
# Configuration paths
config_dir = Path("/workspace/collab-splats/docs/splats/configs")
dataset_name = "birds_date-02062024_video-C0043" #"bicycle_mapanything"

print(f"Loading configuration: {dataset_name}")
print(f"Config directory: {config_dir}")

# Load the splatter configuration
splatter = Splatter.from_config_file(
    dataset=dataset_name,
    config_dir=config_dir,
)

print(f"\n" + "="*70)
print("Configuration Loaded")
print("="*70)
print(f"Dataset: {splatter.config['file_path']}")
print(f"Method: {splatter.config['method']}")
print(f"Input type: {splatter.config['input_type']}")
print(f"Output: {splatter.config['output_path']}")
print("="*70)

## Step 2: Frame Extraction with Optical Flow 

Use the optical flow method to find the optimal frames for video decimation

In [ ]:
output_dir = splatter.config['output_path'] / "preproc" / "images"

optical_flow_params = {
    'min_disparity': 60.0,
    'motion_weight': 0.5,
    'coverage_weight': 0.5,
    'rotation_threshold': 3.0,
    'adaptive_threshold': True,
    'verbose': True
}

# Create a selector with default parameters
selector = OpticalFlowFrameSelector(**optical_flow_params)

print(f"Selector initialized:")
print(f"  Motion weight: {selector.motion_weight:.1%}")
print(f"  Coverage weight: {selector.coverage_weight:.1%}")
print(f"  Min disparity: {selector.min_disparity} pixels")
print(f"  Rotation threshold: {selector.rotation_threshold} degrees")

selected_indices, metrics = selector.process_video(
    video_path=splatter.config['file_path'],
    selection_threshold=0.51,      # Threshold for combined score
    save_selected_frames=True,    # Save selected frames to disk
    output_dir=output_dir,              # Auto-generate output directory
)

print(f"\nProcessed {metrics['total_frames']} frames")
print(f"Selected {metrics['selected_frames']} frames ({metrics['selection_rate']:.1%})")
print(f"Selected frame indices: {selected_indices[:10]}...")  # Show first 10


### Visualize selection metrics + frames

In [ ]:
# Create comprehensive summary plot
fig = create_selection_summary_plot(metrics, figsize=(16, 10))
plt.show()

print("\nInterpretation:")
print("  - Top-left: Combined selection scores over time")
print("  - Top-right: Motion vs coverage component contributions")
print("  - Bottom-left: Distribution of displacement magnitudes")
print("  - Bottom-right: Summary statistics")

## Step 3: Use Feedforward Model

In [ ]:
image_dir = splatter.config['output_path'] / "preproc" / "images"

pipeline_results = run_mapanything_pipeline(
    image_dir=image_dir,
    output_dir=splatter.config['output_path'],
    cleanup_after=True,
    verbose=True,
    create_mesh=True,
)

Test piecewise pipeline

In [ ]:
from mapanything_utils import load_mapanything_model, load_and_preprocess_images

image_dir = splatter.config['output_path'] / "preproc" / "images"
output_dir = splatter.config['output_path']

# Setup directories
preproc_dir = output_dir / "preproc"
colmap_dir = preproc_dir / "colmap"

# Step 1: Load model
model = load_mapanything_model(model_name="facebook/map-anything", verbose=True)

# Step 2: Load images
views, image_paths = load_and_preprocess_images(image_dir, verbose=True)
image_names = [p.name for p in image_paths]

Forward pass through model

In [ ]:
from mapanything_utils import run_mapanything_inference

# Get model dimensions
model_width = views[0]['img'].shape[-1]
model_height = views[0]['img'].shape[-2]

# Step 3: Run inference
outputs = run_mapanything_inference(model, views, verbose=True) #, **kwargs)

Now load the colmap reconstruction (rescaled) and see if we can mesh from that?

In [ ]:
import torch
from typing import List, Dict, Tuple, Any
from mapanything.utils.geometry import depthmap_to_world_frame

def format_outputs_for_visualization(
    outputs: List[Dict],
    filter_black_bg: bool = False,
    filter_white_bg: bool = False,
) -> Tuple[Dict[str, np.ndarray], Any]:
    """
    Convert MapAnything model outputs to format for visualization.
    
    Args:
        outputs: List of prediction dictionaries from model.infer()
        views: List of view dictionaries from load_images()
        high_level_config: Configuration dictionary
        filter_black_bg: Whether to mask black background pixels
        filter_white_bg: Whether to mask white background pixels
    
    Returns:
        Tuple of (predictions dict, processed_data for visualization)
    """

    processed = []
    
    for pred in outputs:
        # Extract tensors
        depth = pred["depth_z"][0].squeeze(-1)
        intrinsic = pred["intrinsics"][0]
        extrinsic = pred["camera_poses"][0]
        image = pred["img_no_norm"][0].cpu().numpy()
        
        # Compute world points
        pts3d_world, valid_depth = depthmap_to_world_frame(depth, intrinsic, extrinsic)
        
        # Build mask
        mask = pred["mask"][0].squeeze(-1).cpu().numpy().astype(bool) if "mask" in pred else np.ones_like(depth.cpu().numpy(), dtype=bool)
        mask = mask & valid_depth.cpu().numpy()
        
        # Background filtering
        if filter_black_bg or filter_white_bg:
            img_uint8 = (image * 255).astype(np.uint8)
            if filter_black_bg:
                mask = mask & (img_uint8.sum(axis=-1) >= 16)
            if filter_white_bg:
                mask = mask & ~((img_uint8 > 240).all(axis=-1))
        
        processed.append({
            "world_points": pts3d_world.cpu().numpy(),
            "images": image,
            "extrinsic": extrinsic.cpu().numpy(),
            "intrinsic": intrinsic.cpu().numpy(),
            "final_mask": mask,
            "depth": depth.cpu().numpy(),
            "conf": pred["conf"][0].squeeze(-1).cpu().numpy(),
        })
    
    # Stack all arrays
    depth_stack = np.stack([p["depth"] for p in processed])
    predictions = {
        "world_points": np.stack([p["world_points"] for p in processed]),
        "images": np.stack([p["images"] for p in processed]),
        "extrinsic": np.stack([p["extrinsic"] for p in processed]),
        "intrinsic": np.stack([p["intrinsic"] for p in processed]),
        "final_mask": np.stack([p["final_mask"] for p in processed]),
        "depth": depth_stack[..., None],  # Add channel dimension
        "conf": np.stack([p["conf"] for p in processed]),
    }
    
    # Clean up GPU memory
    torch.cuda.empty_cache()
    
    return predictions

In [ ]:
preds = format_outputs_for_visualization(
    outputs=outputs,
    filter_black_bg=True,
    filter_white_bg=True,
)

In [ ]:
from mapanything.utils.hf_utils.viz import predictions_to_glb

glbscene = predictions_to_glb(
    preds,
    mask_black_bg=True,
    mask_white_bg=True,
    as_mesh=True,
    conf_percentile=60,
)

In [ ]:
import trimesh

scene = glbscene.dump(concatenate=False)

# 2. Remove tiny helper meshes (14 verts / 48 faces)
meshes = [
    g for g in scene
    if not (g.vertices.shape[0] == 14 and g.faces.shape[0] == 48)
]

print(f"Kept {len(meshes)} meshes")

# 4. Concatenate into one mesh
mesh = trimesh.util.concatenate(meshes)

# 5. Clean
mesh.merge_vertices()
mesh.remove_duplicate_faces()
mesh.remove_degenerate_faces()
mesh.remove_unreferenced_vertices()
mesh.remove_infinite_values()

In [ ]:
import open3d as o3d
import numpy as np
import trimesh

# Convert Trimesh -> Open3D
o3d_mesh = o3d.geometry.TriangleMesh(
    vertices=o3d.utility.Vector3dVector(mesh.vertices),
    triangles=o3d.utility.Vector3iVector(mesh.faces)
)

# Simplify / decimate to ~300k faces (or whatever target)
o3d_mesh = o3d_mesh.simplify_quadric_decimation(target_number_of_triangles=300_000)

# Convert back to Trimesh (if you want to keep using Trimesh)
mesh_simplified = trimesh.Trimesh(
    vertices=np.asarray(o3d_mesh.vertices),
    faces=np.asarray(o3d_mesh.triangles),
    process=False
)
print(mesh_simplified)

In [ ]:
mesh_simplified.show()

## Step 4: Visualize PCD

In [ ]:
point_cloud = pv.PolyData(str(pipeline_results['point_cloud']))
print(f"  Points: {point_cloud.n_points:,}")

# Use collab_splats visualization
pcd_kwargs = MESH_KWARGS.copy()
pcd_kwargs.update({
    "point_size": 1,
    "render_points_as_spheres": True,
    "ambient": 0.3,
    "diffuse": 0.8,
    "specular": 0.1,
})

plotter = visualize_splat(
    mesh=point_cloud,
    mesh_kwargs=pcd_kwargs,
    viz_kwargs=VIZ_KWARGS,
)

plotter.show()